# Imports

In [1]:
from tensorflow import keras
import seaborn as sns
import random
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Nadam
from tensorflow.keras.callbacks import ModelCheckpoint
import pandas as pd
from sklearn.model_selection import train_test_split
import librosa
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
confusion_matrix, classification_report,
accuracy_score, precision_score,
recall_score, f1_score
)
import matplotlib.pyplot as plt

# Functions

In [ ]:
def generate_results(base_data: str, checkpoint_dir: str, save_name: str, classes_ignore: list[str]):
    BASE_DATA = f'../data/{base_data}'
    CHECKPOINT_DIR = f'../checkpoints/{checkpoint_dir}'

    features_type = ['phase', 'mel_db', 'chroma', 'mfcc', 'mfcc_delta', 'cqt_db', 'contrast', 'spec_db']

    all_results = []
    conf_matrices = []
    class_labels_global = None

    for feature in features_type:

        print(f'Avaliando: {feature}')

        base_dir = os.path.join(BASE_DATA, feature)

        filepaths = []
        labels = []

        for label in os.listdir(base_dir):
            class_dir = os.path.join(base_dir, label)
            if os.path.isdir(class_dir):
                for file in os.listdir(class_dir):
                    filepaths.append(os.path.join(class_dir, file))
                    labels.append(label)

        df = pd.DataFrame({
            'filename': filepaths,
            'class': labels
        })
        for class_name in classes_ignore:

            df = df[df['class'] != class_name]

        # df = df[df['class'] != 'Asthma']
        # df = df[df['class'] != 'LRTI']

        train_df, test_df = train_test_split(
            df,
            test_size=0.2,
            stratify=df['class'],
            random_state=42
        )

        datagen = ImageDataGenerator(rescale=1./255)

        test_generator = datagen.flow_from_dataframe(
            test_df,
            x_col='filename',
            y_col='class',
            target_size=(570, 370),
            class_mode='categorical',
            batch_size=16,
            shuffle=False
        )

        model_path = os.path.join(CHECKPOINT_DIR, f'model_{feature}.keras')
        model = tf.keras.models.load_model(model_path)

        preds = model.predict(test_generator, verbose=0)
        y_pred = np.argmax(preds, axis=1)
        y_true = test_generator.classes

        class_labels = list(test_generator.class_indices.keys())

        if class_labels_global is None:
            class_labels_global = class_labels

        acc = accuracy_score(y_true, y_pred)
        precision_global = precision_score(y_true, y_pred, average='weighted')
        recall_global = recall_score(y_true, y_pred, average='weighted')
        f1_global = f1_score(y_true, y_pred, average='weighted')

        report = classification_report(
            y_true,
            y_pred,
            target_names=class_labels,
            output_dict=True
        )

        for cls in class_labels:
            all_results.append({
                'feature': feature,
                'class': cls,
                'precision': report[cls]['precision'],
                'recall': report[cls]['recall'],
                'f1_score': report[cls]['f1-score'],
                'support': report[cls]['support'],
                'accuracy_global': acc,
                'precision_global': precision_global,
                'recall_global': recall_global,
                'f1_global': f1_global
            })

        cm = confusion_matrix(y_true, y_pred)
        conf_matrices.append(cm)

    results_df = pd.DataFrame(all_results)
    results_df.to_excel(f'../results/results_{save_name}.xlsx', index=False)


    n = len(features_type)
    cols = 3
    rows = int(np.ceil(n / cols))

    plt.figure(figsize=(cols * 5, rows * 4))

    for i, (feature, cm) in enumerate(zip(features_type, conf_matrices)):
        plt.subplot(rows, cols, i + 1)

        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            xticklabels=class_labels_global,
            yticklabels=class_labels_global
        )

        plt.title(feature)
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        plt.xlabel('Pred')
        plt.ylabel('True')

    plt.tight_layout()
    plt.savefig(f"../results/confusion_{save_name}.png", dpi=300)
    plt.show()

    print("\n===== Resultados por Classe =====\n")
    results_df.sort_values(by=['feature', 'f1_score'], ascending=[True, False])


# Train InceptionV3

In [4]:
base = '../processed_data_6sec_merged'
features_type = os.listdir(base)

for feature in features_type:
    base_dir = os.path.join(base, feature)

    filepaths = []
    labels = []

    for label in os.listdir(base_dir):
        class_dir = os.path.join(base_dir, label)
        if os.path.isdir(class_dir):
            for file in os.listdir(class_dir):

                filepaths.append(os.path.join(class_dir, file))
                labels.append(label)

    df = pd.DataFrame({
        'filename': filepaths,
        'class': labels
    })

    df = df[df['class'] != 'LRTI']

    train_df, test_df = train_test_split(
        df,
        test_size=0.2,
        stratify=df['class'],
        random_state=42
    )

    datagen = ImageDataGenerator(rescale=1./255)

    train_generator = datagen.flow_from_dataframe(
        train_df,
        x_col='filename',
        y_col='class',
        target_size=(570, 370),
        class_mode='categorical',
        batch_size=16
    )

    test_generator = datagen.flow_from_dataframe(
        test_df,
        x_col='filename',
        y_col='class',
        target_size=(570, 370),
        class_mode='categorical',
        batch_size=16,
        shuffle=False
    )

    pre_trained_model = InceptionV3(
        input_shape=(570, 370, 3),
        include_top=False,
        weights='imagenet'
    )

    for layer in pre_trained_model.layers:
        layer.trainable = False

    last_output = pre_trained_model.get_layer('mixed7').output

    x = Flatten()(last_output)
    x = Dense(256, activation='relu')(x)
    x = Dense(128, activation='relu')(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.2)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.2)(x)

    num_classes = len(train_generator.class_indices)
    x = Dense(num_classes, activation='softmax')(x)

    model = Model(pre_trained_model.input, x)

    model.compile(
        optimizer=Nadam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    checkpoint = ModelCheckpoint(f"checkpoints_6sec/model_{feature}.keras", save_best_only=True)

    history = model.fit(
        train_generator,
        epochs=10,
        validation_data=test_generator,
        callbacks=[checkpoint]
    )

Found 2327 validated image filenames belonging to 7 classes.
Found 582 validated image filenames belonging to 7 classes.
Epoch 1/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 552s 4s/step - accuracy: 0.6261 - loss: 1.1819 - val_accuracy: 0.7320 - val_loss: 0.9013
Epoch 2/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 569s 4s/step - accuracy: 0.7129 - loss: 0.9129 - val_accuracy: 0.7577 - val_loss: 0.7700
Epoch 3/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 567s 4s/step - accuracy: 0.7516 - loss: 0.7700 - val_accuracy: 0.7680 - val_loss: 0.7548
Epoch 4/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 567s 4s/step - accuracy: 0.7903 - loss: 0.6430 - val_accuracy: 0.7938 - val_loss: 0.6752
Epoch 5/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 567s 4s/step - accuracy: 0.8268 - loss: 0.5266 - val_accuracy: 0.7818 - val_loss: 0.6653
Epoch 6/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 559s 4s/step - accuracy: 0.8530 - loss: 0.4313 - val_accuracy: 0.7715 - val_loss: 0.6674
Epoch 7/10
146/146 ━━━━━━━━━━━━━━━━━━━━ 556s 4s/step - accuracy: 0.8857 - loss: 0.3486 - val_accuracy: 0.

KeyboardInterrupt: 

# Generate Results

In [ ]:
generate_results(base_data='processed_data',
                 checkpoint_dir='checkpoints_ICBHI',
                 save_name='icbhi',
                 classes_ignore=['Asthma', 'LRTI'])

generate_results(base_data='processed_data_merged',
                 checkpoint_dir='checkpoints_merged',
                 save_name='merged',
                 classes_ignore=['LRTI'])

generate_results(base_data='processed_data_6sec_merged',
                 checkpoint_dir='checkpoints_6sec_merged',
                 save_name='6sec',
                 classes_ignore=['LRTI'])
